# Example to run VaSST

### Import from VaSST

In [1]:
# import everything from VaSST
from VaSST import *

### Simulating data from $\mathbf{y} = 6\sin(\mathbf{x}_0)\cos(\mathbf{x}_1) + \pmb\epsilon$, where $\pmb\epsilon \sim \mathrm{N}(\pmb 0_n, \sigma^{2}\mathbf{I}_n)$ with $\sigma^{2} = 0.1^{2}$, $n=2000$, and $p=2$.

In [2]:
# ----------------------------
# 1) simulate dataset
# ----------------------------
def simulate_data(n=2000, noise_std=0.1, seed=0, device="cpu", dtype=torch.float32):
    g = torch.Generator(device=device)
    g.manual_seed(seed)

    # features and response generation
    X = (2.0 * torch.pi) * torch.rand(n, 2, generator=g, device=device, dtype=dtype) - torch.pi
    y_clean = 6.0 * torch.sin(X[:, 0]) * torch.cos(X[:, 1])
    y = y_clean + noise_std * torch.randn(n, generator=g, device=device, dtype=dtype)

    return X, y, y_clean

device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.float32

X, y, y_clean = simulate_data(
    n=2000,
    noise_std=0.1,
    seed=0,
    device=device,
    dtype=dtype,
)

### Build the VaSST model

In [3]:
# ----------------------------
# 2) build VaSST model
# ----------------------------
# custom operator set.
ops = make_operator_set(["mul", "sin", "cos"])

model = VaSST(
    n_features=2,
    n_trees=3,        
    depth=3,         
    operators=ops,
    alpha_split=0.95,
    delta0=2.0,
    eta_op_prior=1.0,
    eta_ft_prior=1.0,
    blr_a0=2.0,
    blr_b0=2.0,
    blr_sigma0_scale=1.0,
    tau_e=1.0,
    tau_op=1.0,
    tau_ft=1.0,
    value_clip=1e3,
    use_tanh_clip=True,
    logits_clip=10.0,
    device=torch.device(device),
    dtype=dtype,
).to(device)

### Train the VaSST model

In [4]:
# ----------------------------
# 3) train
# ----------------------------
cfg = TrainConfig(
    lr=5e-5,
    n_steps=1000,
    mc_samples=8,
    grad_clip=1.0,
    tau_start=1.0,
    tau_end=0.5,
    tau_anneal_steps=600,
    jitter=1e-3,
    log_every=100,
    kl_warmup_steps=300,
    kl_start=0.0,
    kl_end=1.0,
)

logs = train_VaSST(
    model=model,
    X=X,
    y=y,
    cfg=cfg,
    include_intercept=True,
    verbose=False,
    print_every=100,
    use_progress_bar=False,
)

### Rank and print top 10 ranked results according to in-sample RMSE from sampled hard symbolic trees

In [5]:
# ----------------------------
# 4) rank hard samples by RMSE (on TRAIN set)
# ----------------------------
expr_top, beta_top = rank_hard_tree_samples_by_rmse(
    model=model,
    X=X,
    y=y,
    n_samples=2000,
    include_intercept=True,
    jitter=1e-3,
    standardize_xy=False,
    top_k=10,
    feature_names=["x0", "x1"],
)


=== TOP samples by RMSE (expressions) ===
 rank  sample_id     rmse  sigma2_mean              tree_0                  tree_1                     tree_2
    1       1218 0.100903     0.030128 cos(sin((x0 * x0)))     (sin(x0) * cos(x1)) ((x0 * sin(x0)) * sin(x1))
    2        511 0.100903     0.030126 (sin(x0) * cos(x1))                 cos(x0)                    sin(x1)
    3       1404 0.101020     0.030152      cos((x1 * x1))   ((x1 * cos(x1)) * x0)        (sin(x0) * cos(x1))
    4       1905 0.101024     0.030151 (sin(x0) * cos(x1))          cos((x0 * x1)) cos(((x1 * x1) * sin(x1)))
    5        808 0.101027     0.030152 (sin(x0) * cos(x1))       cos(sin(sin(x1)))        cos(cos((x1 * x1)))
    6        558 0.101029     0.030152   sin(cos(cos(x0)))     (sin(x0) * cos(x1))        (sin(x0) * sin(x0))
    7         50 0.101037     0.030151 (cos(x1) * sin(x0)) ((x0 * x0) * (x0 * x0))               cos(cos(x1))
    8        382 0.166474     0.052817             sin(x1)          (sin(x1) 